NameError: name '__file__' is not defined

In [6]:
# ─── 2. Create diverse load shapes ───────────────────────────────────────
# 720 hours = 30 days × 24 hours
# Each load gets a slightly different shape so voltage correlations
# are phase-driven, not just common-mode feeder variation.
# We build a realistic residential daily curve and add per-load diversity.

np.random.seed(42)
n_hours = 720
hours_in_day = np.arange(24)

def daily_residential_shape():
    """Realistic residential load shape: morning + evening peaks."""
    base = (
        0.4
        + 0.3 * np.exp(-((hours_in_day - 8)**2) / 8)   # morning peak
        + 0.5 * np.exp(-((hours_in_day - 19)**2) / 6)  # evening peak
        + 0.1 * np.random.randn(24)                      # day-to-day noise
    )
    return np.clip(base, 0.1, 1.2)

# Get all load names
load_names = dss.Loads.AllNames()
print(f"Total loads: {len(load_names)}")

# Build and attach a unique shape to each load
for load_name in load_names:
    # 30-day series by tiling a slightly varied daily curve
    shape_30day = []
    for day in range(30):
        shape_30day.extend(daily_residential_shape())
    shape_array = np.array(shape_30day)

    # Define the LoadShape object in OpenDSS
    shape_cmd = (
        f"New LoadShape.shape_{load_name} "
        f"npts={n_hours} interval=1 "
        f"mult=({','.join(f'{v:.4f}' for v in shape_array)})"
    )
    dss.Command(shape_cmd)

    # Attach it to this load
    dss.Command(f"Edit Load.{load_name} daily=shape_{load_name}")

print("Load shapes attached.")

Total loads: 91
Load shapes attached.


C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\1243134733.py:39: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command(shape_cmd)
C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\1243134733.py:42: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command(f"Edit Load.{load_name} daily=shape_{load_name}")


In [7]:
# ─── 3. Add a few PV systems ─────────────────────────────────────────────
# PV irradiance shape: sunrise ~6, peak ~12, sunset ~18
def daily_pv_shape():
    pv = np.zeros(24)
    for h in range(6, 19):
        pv[h] = max(0, np.sin(np.pi * (h - 6) / 12))
    pv += np.clip(0.05 * np.random.randn(24), -0.05, 0.05)  # cloud variation
    return np.clip(pv, 0, 1)

pv_buses = ['66', '44', '11', '83', '34']   # spread across feeder phases
pv_kw    = [50,   60,   40,   70,   45]

for i, (bus, kw) in enumerate(zip(pv_buses, pv_kw)):
    # Build 30-day PV shape
    pv_30day = []
    for day in range(30):
        pv_30day.extend(daily_pv_shape())
    pv_array = np.array(pv_30day)

    shape_cmd = (
        f"New LoadShape.pvshape_{i} "
        f"npts={n_hours} interval=1 "
        f"mult=({','.join(f'{v:.4f}' for v in pv_array)})"
    )
    dss.Command(shape_cmd)

    dss.Command(
        f"New PVSystem.pv_{i} bus1={bus} phases=1 "
        f"kVA={kw} Pmpp={kw} kV=2.4 "
        f"daily=pvshape_{i}"
    )

print("PV systems added.")

PV systems added.


C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\1066774971.py:25: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command(shape_cmd)
C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\1066774971.py:27: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command(


In [8]:
# ─── 4. Run QSTS and record measurements ─────────────────────────────────
dss.run_command("Set Mode=Daily")
dss.run_command("Set StepSize=1h")
dss.run_command("Set Number=1")    # solve one step at a time so we can record

bus_names   = dss.Circuit.AllBusNames()
n_buses     = len(bus_names)

# Storage arrays
volt_records = []   # voltage magnitudes (pu) per timestep
pq_records   = []   # P and Q per load per timestep
timestamps   = []

print("Running QSTS for 720 hours...")
for hour in range(n_hours):
    dss.run_command("Solve")

    # Voltage magnitudes at all buses (per unit, one value per node)
    # AllBusMagPu returns a flat list ordered by AllBusNames
    vmag = np.array(dss.Circuit.AllBusMagPu())
    volt_records.append(vmag)

    # Per-load P and Q
    load_pq = {}
    for load_name in load_names:
        dss.Loads.Name(load_name)
        load_pq[load_name] = (dss.CktElement.Powers()[0],    # P in kW
                               dss.CktElement.Powers()[1])    # Q in kvar
    pq_records.append(load_pq)
    timestamps.append(hour)

    if hour % 100 == 0:
        print(f"  Hour {hour}/720 done")

print("QSTS complete.")

# Convert voltage records to DataFrame: rows = hours, cols = bus nodes
volt_df = pd.DataFrame(volt_records)
volt_df.index = timestamps
print(f"Voltage matrix shape: {volt_df.shape}")   # should be (720, ~278)

C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\2984395629.py:2: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command("Set Mode=Daily")
C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\2984395629.py:3: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command("Set StepSize=1h")
C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\2984395629.py:4: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command("Set Number=1")    # solve one step at a time so we can record
C:\Users\C838122727\AppData\Local\Temp\ipykernel_23132\2984395629.py:16: DeprecationWarning: run_command is deprecated (use Command, Commands or th

Running QSTS for 720 hours...
  Hour 0/720 done
  Hour 100/720 done
  Hour 200/720 done
  Hour 300/720 done
  Hour 400/720 done
  Hour 500/720 done
  Hour 600/720 done
  Hour 700/720 done
QSTS complete.
Voltage matrix shape: (720, 279)


In [10]:
# ─── 5. Extract ground-truth phase labels ────────────────────────────────
import re

loads_dss_path = r"C:\Users\C838122727\Documents\CSU\research\Summer_2026\123Bus\IEEE123Loads.DSS"  # update this

load_phase_labels = {}   # load_name -> phase label 'A', 'B', or 'C'

node_to_phase = {1: 'A', 2: 'B', 3: 'C'}

with open(loads_dss_path, 'r') as f:
    for line in f:
        m = re.match(r'New Load\.(\S+)\s+Bus1=(\S+)', line, re.IGNORECASE)
        if m:
            load_name = m.group(1)
            bus_raw   = m.group(2)
            # Node number is after the dot: "1.1" → node 1 → Phase A
            parts = bus_raw.split('.')
            node  = int(parts[1]) if len(parts) > 1 else 1
            load_phase_labels[load_name] = node_to_phase.get(node, 'Unknown')

# Sanity check
phase_counts = {'A': 0, 'B': 0, 'C': 0}
for v in load_phase_labels.values():
    phase_counts[v] += 1
print("Ground truth phase counts:", phase_counts)

# Save to CSV for Week 2
labels_df = pd.DataFrame.from_dict(
    load_phase_labels, orient='index', columns=['true_phase']
)
labels_df.index.name = 'load_name'
labels_df.to_csv('ground_truth_phases.csv')
print("Saved ground_truth_phases.csv")
print(labels_df.head(10))

Ground truth phase counts: {'A': 40, 'B': 23, 'C': 28}
Saved ground_truth_phases.csv
          true_phase
load_name           
S1a                A
S2b                B
S4c                C
S5c                C
S6c                C
S7a                A
S9a                A
S10a               A
S11a               A
S12b               B


In [11]:
# ─── 6. Save voltage data for Week 2 ─────────────────────────────────────
volt_df.to_csv('voltage_timeseries.csv')

# Also save a per-load voltage series (matching loads to their bus)
# This is what Week 2 baseline and GNN both use as input features
load_bus_map = {}
for load_name in load_names:
    dss.Loads.Name(load_name)
    bus = dss.CktElement.BusNames()[0].split('.')[0].upper()
    load_bus_map[load_name] = bus

load_bus_df = pd.DataFrame.from_dict(
    load_bus_map, orient='index', columns=['bus']
)
load_bus_df.to_csv('load_bus_map.csv')
print("Saved voltage_timeseries.csv and load_bus_map.csv")

Saved voltage_timeseries.csv and load_bus_map.csv
